In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(*(['..' + os.sep] * 1))))
from morning.pipeline.converter import dt
from morning_server import message
from datetime import datetime, timedelta, date, time
from pymongo import MongoClient
from configs import db
from morning.back_data import holidays
holidays.read_holiday_excel()

In [3]:
%run ticktracker.ipynb

In [4]:
FRAME_COUNT = 3

def get_trend_frame(second_ticks):
    trend_frame = int(len(second_ticks) / FRAME_COUNT)
    if trend_frame % 2 != 0:
        trend_frame -= 1
    return trend_frame

In [5]:
def is_support_up_trend(points):
    low_prices = []
    prev_price = 0

    for p in points:
        if p[1] == prev_price:
            continue
        low_prices.append(p[1])
        prev_price = p[1]
    
    if len(low_prices) >= 2:
        prev_price = 0
        for l in low_prices:
            if l < prev_price:
                return False
            prev_price = l
        return True
    return False


def is_resistance_up_trend(points):
    if len(points) < 2:
        return False

    arr_time = np.arange(0, len(points))
    prices = [p[1] for p in points]
    m, _ = np.polyfit(arr_time, prices, 1)
    return m > 0

In [6]:
def get_tick_data_by_datetime(code, from_datetime, until_datetime):
    _mongo_collection = MongoClient(db.HOME_MONGO_ADDRESS).trade_alarm
    data = list(_mongo_collection[code].find({'date': {'$gte': from_datetime, '$lte': until_datetime}}))
    converted_data = []
    for td in data:
        if code.endswith(message.BIDASK_SUFFIX):
            converted = dt.cybos_stock_ba_tick_convert(td)
        else:
            converted = dt.cybos_stock_tick_convert(td)
        converted_data.append(converted)
    return converted_data

In [7]:
def get_yesterday_data(code, search_date):
    yesterday = holidays.get_yesterday(search_date)
    _mongo_collection = MongoClient(db.HOME_MONGO_ADDRESS).stock
    d = yesterday.year * 10000 + yesterday.month * 100 + yesterday.day
    data = list(_mongo_collection[code + '_D'].find({'0': d}))
    return dt.cybos_stock_day_tick_convert(data[0])

In [24]:
def get_yesterday_min_data(code, search_date):
    yesterday = holidays.get_yesterday(search_date)
    print('yesterday', yesterday)
    _mongo_collection = MongoClient(db.HOME_MONGO_ADDRESS).stock
    d = yesterday.year * 10000 + yesterday.month * 100 + yesterday.day
    data = list(_mongo_collection[code + '_M'].find({'0': d}))
    converted_data = []
    for md in data:
        converted = dt.cybos_stock_day_tick_convert(md)
        converted['code'] = code
        converted['date'] = datetime.combine(yesterday, time(int(converted['time'] / 100), int(converted['time'] % 100)))
        converted_data.append(converted)

    return converted_data

In [9]:
def get_support_points(second_ticks):
    trend_frame = get_trend_frame(second_ticks)
    half = int(trend_frame / 2)
    is_support = np.full((len(second_ticks),), False)

    supports = []

    for i, stick in enumerate(second_ticks):
        left_index = i - half
        right_index = left_index + trend_frame
        if left_index < 0:
            left_index = 0
        if right_index > len(second_ticks):
            right_index = len(second_ticks)
        
        arr = second_ticks[left_index:right_index]
        min_price = min([st.low for st in arr])
        if stick.low == min_price and not np.any(is_support[left_index:right_index]):
            is_support[i] = True
            supports.append((stick.time, stick.low))
    return is_support, supports

In [10]:
def get_resistance_points(second_ticks):
    trend_frame = get_trend_frame(second_ticks)
    half = int(trend_frame / 2)
    is_resistance = np.full((len(second_ticks),), False)

    resistances = []

    for i, stick in enumerate(second_ticks):
        left_index = i - half
        right_index = left_index + trend_frame
        if left_index < 0:
            left_index = 0
        if right_index > len(second_ticks):
            right_index = len(second_ticks)
        
        arr = second_ticks[left_index:right_index]
        max_price = max([st.high for st in arr])
        if stick.high == max_price and not np.any(is_resistance[left_index:right_index]):
            is_resistance[i] = True
            resistances.append((stick.time, stick.high))

    return is_resistance, resistances

In [87]:
code = 'A047400'
search_date = date(2020, 7, 1)

In [88]:
ydata = get_yesterday_data(code, search_date)
ymindata = get_yesterday_min_data(code, search_date)
tick_data = get_tick_data_by_datetime(code,
                                      datetime.combine(search_date, time(8, 59)),
                                      datetime.combine(search_date, time(9, 50)))
print(len(ymindata))

yesterday 2020-06-30
0


In [89]:
def second_callback(second_ticks, tick_time, ask_price):
    pass

In [90]:
tick_tracker = TickTracker(code, ydata['close_price'], second_callback)

In [91]:
for t in tick_data:
    tick_tracker.handle_tick(t)

In [92]:
def find_index(dt, st):
    min_seconds = 10000
    near_index = 0
    for i, stick in enumerate(st):
        t_order = (stick.time, dt) if stick.time > dt else (dt, stick.time)
        if (t_order[0] - t_order[1]).seconds < min_seconds:
            min_seconds = (t_order[0] - t_order[1]).seconds
            near_index = i
    return near_index

In [97]:
#limit_index = find_index(datetime(2020, 7, 1, 9, 5, 12, 666000), tick_tracker.second_ticks)#len(tick_tracker.second_ticks)
limit_index = len(tick_tracker.second_ticks)
second_ticks = tick_tracker.second_ticks[:limit_index]
second_ticks[-1].time
datetime_arr = [s.time for s in second_ticks]
low_arr = [s.low for s in second_ticks]
high_arr = [s.high for s in second_ticks]
close_arr = [s.close for s in second_ticks]

In [98]:
timeframe_secs = (datetime_arr[-1] - datetime_arr[0]).seconds

In [99]:
resistance, rpoints = get_resistance_points(second_ticks)
support, spoints = get_support_points(second_ticks)
print(rpoints, is_resistance_up_trend(rpoints))
print(spoints, is_support_up_trend(spoints))

[(datetime.datetime(2020, 7, 1, 9, 1, 22, 322000), 4075), (datetime.datetime(2020, 7, 1, 9, 27, 9, 91000), 3920), (datetime.datetime(2020, 7, 1, 9, 42, 56, 776000), 3995)] False
[(datetime.datetime(2020, 7, 1, 9, 0, 43, 10000), 3895), (datetime.datetime(2020, 7, 1, 9, 16, 7, 588000), 3875), (datetime.datetime(2020, 7, 1, 9, 31, 47, 806000), 3860), (datetime.datetime(2020, 7, 1, 9, 49, 37, 808000), 3895)] False


In [100]:
from plotly.subplots import make_subplots
open_y = np.full((len(datetime_arr),), tick_tracker.open)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=datetime_arr, y=close_arr, line=dict(color='black')), secondary_y=False)
fig.add_trace(go.Scatter(x=datetime_arr, y=open_y, line=dict(color='magenta')), secondary_y=False)
fig.add_trace(go.Scatter(x=datetime_arr, y=support, opacity=0.3, line=dict(color='blue')), secondary_y=True,)
fig.add_trace(go.Scatter(x=datetime_arr, y=resistance, opacity=0.3, line=dict(color='red')), secondary_y=True,)
fig.show()

In [40]:
open_y = np.full((len(datetime_arr),), tick_tracker.open)
fig = make_subplots()
ymindata_date_arr = [ym['date'] for ym in ymindata]
ymindata_price_arr = [ym['close_price'] for ym in ymindata]
fig.add_trace(go.Scatter(x=ymindata_date_arr, y=ymindata_price_arr, line=dict(color='black')), secondary_y=False)
print((ydata['close_price'] - ydata['start_price']) / ydata['start_price'] * 100.)
fig.show()


9.939148073022313
